In [2]:
import subprocess
import sys

# Install Prophet (PyPI package name: 'prophet')
packages = ['prophet']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ Prophet installed (or already present)")

✓ Prophet installed (or already present)


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)

# --- STEP 1: LOAD DATA ---
FILE_NAME = 'energy_data_3_years.csv'
try:
    data = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"Error: The file '{FILE_NAME}' was not found.")
    exit()

# Prepare data
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date').reset_index(drop=True)

print("Data loaded successfully!")
print(f"Date range: {data['Date'].min().date()} to {data['Date'].max().date()}")
print(f"Total records: {len(data)}")

# --- STEP 2: PREPARE SOLAR DATA ---
solar_col = 'Solar_Energy_Produced_kWh'
solar_data = data[['Date', solar_col]].copy()
solar_data.columns = ['ds', 'y']
solar_data['y'] = solar_data['y'].fillna(solar_data['y'].mean())

print(f"\n{'='*70}")
print("SOLAR ENERGY - HISTORICAL STATISTICS")
print(f"{'='*70}")
print(f"Mean: {solar_data['y'].mean():.2f} kWh")
print(f"Std: {solar_data['y'].std():.2f} kWh")
print(f"Min: {solar_data['y'].min():.2f} kWh")
print(f"Max: {solar_data['y'].max():.2f} kWh")

# --- STEP 3: TRAIN PROPHET MODEL ---
print(f"\n{'='*70}")
print("TRAINING PROPHET MODEL (SOLAR)")
print(f"{'='*70}")

solar_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive',
    interval_width=0.95,
    changepoint_prior_scale=0.05
)

solar_model.fit(solar_data)
print("✓ Solar model trained successfully!")

# --- STEP 4: FORECAST SOLAR FOR 2026 ---
print(f"\nGenerating 365-day forecast for 2026...")
future_dates = solar_model.make_future_dataframe(periods=365)
solar_forecast = solar_model.predict(future_dates)

# Get only 2026 predictions
solar_2026 = solar_forecast[solar_forecast['ds'] >= '2026-01-01'].copy()
solar_2026 = solar_2026[solar_2026['ds'] <= '2026-12-31'].copy()

# Ensure positive values
solar_2026['yhat'] = np.maximum(solar_2026['yhat'].values, 0)
solar_2026['yhat_lower'] = np.maximum(solar_2026['yhat_lower'].values, 0)

print(f"✓ Forecast generated!")
print(f"\nSolar Forecast Statistics (2026):")
print(f"Mean: {solar_2026['yhat'].mean():.2f} kWh")
print(f"Std: {solar_2026['yhat'].std():.2f} kWh")
print(f"Min: {solar_2026['yhat'].min():.2f} kWh")
print(f"Max: {solar_2026['yhat'].max():.2f} kWh")

# Create output dataframe
forecast_df = pd.DataFrame({
    'Date': solar_2026['ds'].values,
    'Solar_Energy_Forecast_kWh': solar_2026['yhat'].values,
    'Solar_Lower_CI': solar_2026['yhat_lower'].values,
    'Solar_Upper_CI': solar_2026['yhat_upper'].values,
    'Trend': solar_2026['trend'].values,
    'Yearly_Seasonality': solar_2026['yearly'].values if 'yearly' in solar_2026.columns else 0,
    'Weekly_Seasonality': solar_2026['weekly'].values if 'weekly' in solar_2026.columns else 0
})

# --- STEP 5: WIND ENERGY FORECAST ---
wind_col = 'Wind_Energy_Produced_kWh'
if wind_col in data.columns:
    print(f"\n{'='*70}")
    print("WIND ENERGY - PROPHET FORECAST")
    print(f"{'='*70}")
    
    wind_data = data[['Date', wind_col]].copy()
    wind_data.columns = ['ds', 'y']
    wind_data['y'] = wind_data['y'].fillna(wind_data['y'].mean())
    
    print(f"Mean: {wind_data['y'].mean():.2f} kWh")
    print(f"Std: {wind_data['y'].std():.2f} kWh")
    
    # Train wind model
    print("\nTraining Prophet model for wind...")
    wind_model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='additive',
        interval_width=0.95
    )
    wind_model.fit(wind_data)
    
    # Forecast wind
    wind_future = wind_model.make_future_dataframe(periods=365)
    wind_forecast = wind_model.predict(wind_future)
    
    # Get 2026 predictions
    wind_2026 = wind_forecast[wind_forecast['ds'] >= '2026-01-01'].copy()
    wind_2026 = wind_2026[wind_2026['ds'] <= '2026-12-31'].copy()
    wind_2026['yhat'] = np.maximum(wind_2026['yhat'].values, 0)
    wind_2026['yhat_lower'] = np.maximum(wind_2026['yhat_lower'].values, 0)
    
    # Add to forecast dataframe
    forecast_df['Wind_Energy_Forecast_kWh'] = wind_2026['yhat'].values
    forecast_df['Wind_Lower_CI'] = wind_2026['yhat_lower'].values
    forecast_df['Wind_Upper_CI'] = wind_2026['yhat_upper'].values
    
    print(f"✓ Wind forecast completed!")
    print(f"Mean: {wind_2026['yhat'].mean():.2f} kWh")
    print(f"Max: {wind_2026['yhat'].max():.2f} kWh")

# --- STEP 6: SAVE AND DISPLAY RESULTS ---
print(f"\n{'='*70}")
print("FORECAST RESULTS")
print(f"{'='*70}")

# Save forecast
output_file = 'prophet_forecast_2026.csv'
forecast_df[['Date', 'Solar_Energy_Forecast_kWh', 'Solar_Lower_CI', 'Solar_Upper_CI'] + 
             ['Wind_Energy_Forecast_kWh', 'Wind_Lower_CI', 'Wind_Upper_CI'] if 'Wind_Energy_Forecast_kWh' in forecast_df.columns else []].to_csv(output_file, index=False)
print(f"✓ Forecast saved to '{output_file}'")

# Display sample
print(f"\nForecast Sample (First 10 days):")
cols_to_display = ['Date', 'Solar_Energy_Forecast_kWh']
if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    cols_to_display.append('Wind_Energy_Forecast_kWh')
print(forecast_df[cols_to_display].head(10).to_string(index=False))

# Monthly summary
forecast_df['Month'] = pd.to_datetime(forecast_df['Date']).dt.to_period('M')
monthly = forecast_df.groupby('Month')['Solar_Energy_Forecast_kWh'].agg(['mean', 'min', 'max', 'sum'])

print(f"\n{'='*70}")
print("MONTHLY SUMMARY (2026)")
print(f"{'='*70}")
print(monthly.to_string())

# Total energy
total_solar = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"\nTotal Solar Energy (2026): {total_solar:,.2f} kWh")

if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    total_wind = forecast_df['Wind_Energy_Forecast_kWh'].sum()
    total_combined = total_solar + total_wind
    print(f"Total Wind Energy (2026): {total_wind:,.2f} kWh")
    print(f"Combined Total (2026): {total_combined:,.2f} kWh")
    print(f"\nSolar: {(total_solar/total_combined)*100:.1f}%")
    print(f"Wind: {(total_wind/total_combined)*100:.1f}%")

print(f"\n{'='*70}")
print("✓ PROPHET FORECASTING COMPLETE")
print(f"{'='*70}")

Importing plotly failed. Interactive plots will not work.


Data loaded successfully!
Date range: 2022-12-10 to 2025-12-08
Total records: 1095

SOLAR ENERGY - HISTORICAL STATISTICS
Mean: 9.53 kWh
Std: 2.01 kWh
Min: 4.50 kWh
Max: 14.52 kWh

TRAINING PROPHET MODEL (SOLAR)


15:58:40 - cmdstanpy - INFO - Chain [1] start processing
15:58:41 - cmdstanpy - INFO - Chain [1] done processing
15:58:41 - cmdstanpy - INFO - Chain [1] done processing


✓ Solar model trained successfully!

Generating 365-day forecast for 2026...
✓ Forecast generated!

Solar Forecast Statistics (2026):
Mean: 9.78 kWh
Std: 1.76 kWh
Min: 6.78 kWh
Max: 12.29 kWh

WIND ENERGY - PROPHET FORECAST
Mean: 3.93 kWh
Std: 3.69 kWh

Training Prophet model for wind...
✓ Forecast generated!

Solar Forecast Statistics (2026):
Mean: 9.78 kWh
Std: 1.76 kWh
Min: 6.78 kWh
Max: 12.29 kWh

WIND ENERGY - PROPHET FORECAST
Mean: 3.93 kWh
Std: 3.69 kWh

Training Prophet model for wind...


15:58:42 - cmdstanpy - INFO - Chain [1] start processing
15:58:42 - cmdstanpy - INFO - Chain [1] done processing
15:58:42 - cmdstanpy - INFO - Chain [1] done processing


✓ Wind forecast completed!
Mean: 4.32 kWh
Max: 9.17 kWh

FORECAST RESULTS
✓ Forecast saved to 'prophet_forecast_2026.csv'

Forecast Sample (First 10 days):
      Date  Solar_Energy_Forecast_kWh  Wind_Energy_Forecast_kWh
2026-01-01                   7.111736                  1.659477
2026-01-02                   7.202726                  1.603803
2026-01-03                   7.142118                  1.555637
2026-01-04                   7.060636                  1.515037
2026-01-05                   7.127186                  1.481917
2026-01-06                   7.153371                  1.456054
2026-01-07                   7.241509                  1.437092
2026-01-08                   7.126474                  1.424563
2026-01-09                   7.217762                  1.417895
2026-01-10                   7.159337                  1.416432

MONTHLY SUMMARY (2026)
              mean        min        max         sum
Month                                               
2026-01   